In [1]:
import pathlib
import sys
import matplotlib.pyplot as plt
from tqdm import tqdm
import xarray as xr
import importlib
import numpy as np
import ipywidgets as widgets
script_dir = pathlib.Path().parent.absolute()
parent_dir = script_dir.parents[0]
sys.path.append(str(parent_dir))
from dissipationSML import reading, plotting, tools, interactive

In [2]:
importlib.reload(plotting)
importlib.reload(interactive)

<module 'dissipationSML.interactive' from '/Users/tillmoritz/Desktop/Master_thesis/Git/dissipationSML/dissipationSML/interactive.py'>

In [ ]:
# Example usage
yaml_path = str(parent_dir) + '/dissipationSML/glider_server.yaml'  # Update with your actual path
selected_glider = interactive.interactive_glider_selection(yaml_path)

Dropdown(description='Select Glider:', options=('005', '012', '014', '016', '101', '102', '103', '104', '105')…

Dropdown(description='Select Mission:', options=('06/08 (dives: 386)', '06/09 (dives: 464)', '11/08 (dives: 48…

Button(description='Confirm Selection', style=ButtonStyle())

Selected Path: https://www.ncei.noaa.gov/data/oceans/glider/seaglider/uw/005/20080606/


In [126]:
data_path = selected_glider['path']
end_profile = selected_glider['dives']
destination_folder = "/Users/tillmoritz/Desktop/Master_thesis/Data/Glider"
mission_path = destination_folder + '/' + selected_glider['glider'] + '/' + selected_glider['mission']

In [127]:
ds_glider = xr.open_dataset(mission_path + '/all_data_OG1.nc')

In [128]:
ds_glider

<xarray.Dataset> Size: 35MB
Dimensions:                 (N_MEASUREMENTS: 332364)
Coordinates:
    LONGITUDE               (N_MEASUREMENTS) float64 3MB ...
    LATITUDE                (N_MEASUREMENTS) float64 3MB ...
    TIME                    (N_MEASUREMENTS) datetime64[ns] 3MB ...
    DEPTH                   (N_MEASUREMENTS) float64 3MB ...
Dimensions without coordinates: N_MEASUREMENTS
Data variables: (12/15)
    GLIDER_VERT_VELO_MODEL  (N_MEASUREMENTS) float32 1MB ...
    THETA                   (N_MEASUREMENTS) float32 1MB ...
    TEMP                    (N_MEASUREMENTS) float32 1MB ...
    GLIDE_SPEED             (N_MEASUREMENTS) float32 1MB ...
    SIGTHETA                (N_MEASUREMENTS) float32 1MB ...
    SIGMA_T                 (N_MEASUREMENTS) float32 1MB ...
    ...                      ...
    DIVE_NUMBER             (N_MEASUREMENTS) int32 1MB ...
    PROFILE_NUMBER          (N_MEASUREMENTS) float64 3MB ...
    PSAL_RAW                (N_MEASUREMENTS) float32 1MB ...
    TEMP_RAW                (N_MEASUREMENTS) float32 1MB ...
    SIGTHETA_RAW            (N_MEASUREMENTS) float64 3MB ...
    SIGMA_T_RAW             (N_MEASUREMENTS) float64 3MB ...
Attributes: (12/39)
    title:                                      OceanGliders trajectory file
    id:                                         sg005_20080606T180738_delayed
    platform:                                   sub-surface gliders
    platform_vocabulary:                        https://vocab.nerc.ac.uk/coll...
    naming_authority:                           edu.washington.apl
    institution:                                School of Oceanography\nUnive...
    ...                                         ...
    acknowledgment:                             National Science Foundation, ...
    license:                                    These data may be redistribut...
    disclaimer:                                 Data provided AS-IS.
    file_version:                               2.71
    keywords:                                   Water Temperature, Conductivi...
    keywords_vocabulary:                        NASA/GCMD Earth Science Keywo...

In [129]:
dive_slider = widgets.IntSlider(value=0, min=1, max=end_profile/2, step=1, description='Dive number:')
interactive.interactive_dive_profile(ds_glider, dive_slider=dive_slider)

interactive(children=(IntSlider(value=1, description='Dive number:', max=193, min=1), Output()), _dom_classes=…

In [7]:
basepath = '/Users/tillmoritz/Desktop/Master_thesis/Data/ERA-5/'
fluxes_path = 'data_stream-oper_stepType-accum.nc' # surface fluxes (evaporation/precipitation and heat fluxes)
wind_path = 'data_stream-oper_stepType-instant.nc' # wind speed and friction velocity
wave_path = 'data_stream-wave_stepType-instant.nc' # waves (stokes drift, significant wave height and wave period)
years = ['2008']#, '2007', '2008', '2009']

In [20]:
### open all datasets for each year, but cut to area of interest 
### Area of interest: 60°N-62°N, 8°W-10°E
### for each year, open the datasets and cut them to the area of interest
### then concatenate them along the time dimension
for year in years:
    print(year)
    fluxes = xr.open_dataset(basepath + year + '/' + fluxes_path)
    wind = xr.open_dataset(basepath + year + '/' + wind_path)
    wave = xr.open_dataset(basepath + year + '/' + wave_path)
    all_data = xr.concat([fluxes, wind, wave], dim='time')
    ### cut out only on point per day
    all_data.sel()
    #all_data.sel(latitude=slice(60, 61), longitude=slice(-8, -9))
    if year == '2008':
        ds_ERA5 = all_data
    else:
        ds_ERA5 = xr.concat([ds_ERA5,all_data], dim='time')

2008


In [21]:
ds_ERA5

<xarray.Dataset> Size: 1GB
Dimensions:     (time: 3, valid_time: 8784, latitude: 21, longitude: 49)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 70kB 2008-01-01 ... 2008-12-31T23...
  * latitude    (latitude) float64 168B 60.0 60.25 60.5 ... 64.5 64.75 65.0
  * longitude   (longitude) float64 392B -15.0 -14.75 -14.5 ... -3.5 -3.25 -3.0
    number      int64 8B 0
    expver      (valid_time) <U4 141kB '0001' '0001' '0001' ... '0001' '0001'
Dimensions without coordinates: time
Data variables: (12/13)
    tp          (time, valid_time, latitude, longitude) float32 108MB 9.06e-0...
    slhf        (time, valid_time, latitude, longitude) float32 108MB -6.204e...
    ssr         (time, valid_time, latitude, longitude) float32 108MB 3.6e-12...
    str         (time, valid_time, latitude, longitude) float32 108MB -1.234e...
    sshf        (time, valid_time, latitude, longitude) float32 108MB -9.911e...
    e           (time, valid_time, latitude, longitude) float32 108MB -0.0002...
    ...          ...
    v10         (time, valid_time, latitude, longitude) float32 108MB nan ......
    zust        (time, valid_time, latitude, longitude) float32 108MB nan ......
    swh         (time, valid_time, latitude, longitude) float32 108MB nan ......
    pp1d        (time, valid_time, latitude, longitude) float32 108MB nan ......
    ust         (time, valid_time, latitude, longitude) float32 108MB nan ......
    vst         (time, valid_time, latitude, longitude) float32 108MB nan ......
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-01-31T16:47 GRIB to CDM+CF via cfgrib-0.9.1...

In [111]:
mean_time = ds_glider.TIME.groupby(ds_glider.PROFILE_NUMBER).mean()
mean_lon = ds_glider.LONGITUDE.groupby(ds_glider.PROFILE_NUMBER).mean()
mean_lat = ds_glider.LATITUDE.groupby(ds_glider.PROFILE_NUMBER).mean()

tolerance = None
#tolerance = 1.0 # 1-degree radius around the glider position
profiles = np.unique(ds_glider.PROFILE_NUMBER)
### if there is a tolerance, then select and mean the ERA5 data within the tolerance
if tolerance:
    ds_matched_all = []
    for profile in tqdm(profiles):
        lon = mean_lon.sel(PROFILE_NUMBER=profile,drop=True).values
        lat = mean_lat.sel(PROFILE_NUMBER=profile,drop=True).values
        time = mean_time.sel(PROFILE_NUMBER=profile,drop=True).values
        ds_matched = ds_ERA5.sel(valid_time=time, method="nearest")
        ds_matched = ds_matched.sel(longitude=slice(lon - tolerance, lon + tolerance), latitude=slice(lat - tolerance, lat + tolerance))
        #ds_matched = ds_matched.sel(longitude = lon, latitude = lat, method="nearest")
        ds_matched = ds_matched.mean(dim='time').mean(dim='latitude').mean(dim='longitude')
        ### add the profile number to the dataset
        ds_matched['PROFILE_NUMBER'] = profile
        ## add longitude and latitude to the dataset as coordinates
        ds_matched = ds_matched.assign_coords(longitude=lon, latitude=lat)
        ds_matched_all.append(ds_matched)
    ds_matched = xr.concat(ds_matched_all, dim='PROFILE_NUMBER')
    ### add the attributes to the dataset
    ds_matched.attrs = ds_ERA5.attrs
    ### add the attributes of each variable to the dataset
else:
    #ds_matched = ds_ERA5.sel(longitude=mean_lon, latitude=mean_lat, method="nearest")
    ds_matched = ds_ERA5.sel(valid_time=mean_time, longitude=mean_lon, latitude=mean_lat, method="nearest")
    ds_matched = ds_matched.mean(dim='time')
    ### add the attributes of each variable to the dataset
for var in ds_ERA5.variables:
    ds_matched[var].attrs = ds_ERA5[var].attrs


In [116]:
import xarray as xr
import numpy as np
from tqdm import tqdm

def match_era5_to_glider(ds_glider, ds_ERA5, lon_range=None, lat_range=None):
    """
    Matches ERA5 data to glider profiles, either using nearest points or averaging within a spatial range.

    Parameters
    ----------
    ds_glider : xarray.Dataset
        The dataset containing glider data.
    ds_ERA5 : xarray.Dataset
        The ERA5 dataset to be matched.
    lon_range : float or None, optional
        The range (in degrees) for longitude to average ERA5 data. If None, selects the nearest point.
    lat_range : float or None, optional
        The range (in degrees) for latitude to average ERA5 data. If None, selects the nearest point.

    Returns
    -------
    xarray.Dataset
        The matched ERA5 data with the same PROFILE_NUMBER dimension as the glider dataset.
    """
    # Compute mean time, longitude, and latitude per profile
    mean_time = ds_glider.TIME.groupby(ds_glider.PROFILE_NUMBER).mean()
    mean_lon = ds_glider.LONGITUDE.groupby(ds_glider.PROFILE_NUMBER).mean()
    mean_lat = ds_glider.LATITUDE.groupby(ds_glider.PROFILE_NUMBER).mean()

    profiles = np.unique(ds_glider.PROFILE_NUMBER)

    if lon_range or lat_range:
        ds_matched_all = []
        for profile in tqdm(profiles, desc="Matching profiles"):
            lon = mean_lon.sel(PROFILE_NUMBER=profile, drop=True).values
            lat = mean_lat.sel(PROFILE_NUMBER=profile, drop=True).values
            time = mean_time.sel(PROFILE_NUMBER=profile, drop=True).values

            # Select nearest valid_time
            ds_matched = ds_ERA5.sel(valid_time=time, method="nearest")

            # Select longitude and latitude range if provided
            if lon_range:
                ds_matched = ds_matched.sel(longitude=slice(lon - lon_range, lon + lon_range))
            if lat_range:
                ds_matched = ds_matched.sel(latitude=slice(lat - lat_range, lat + lat_range))

            # Compute mean over the selected region
            ds_matched = ds_matched.mean(dim=["latitude", "longitude", "time"], skipna=True)

            # Add profile number, longitude, and latitude as coordinates
            ds_matched = ds_matched.assign_coords(PROFILE_NUMBER=profile, longitude=lon, latitude=lat)
            ds_matched_all.append(ds_matched)

        # Concatenate all profiles along PROFILE_NUMBER
        ds_matched = xr.concat(ds_matched_all, dim="PROFILE_NUMBER")
    else:
        # Match without tolerance (nearest selection)
        ds_matched = ds_ERA5.sel(valid_time=mean_time, longitude=mean_lon, latitude=mean_lat, method="nearest")
        ds_matched = ds_matched.mean(dim="time")

    # Preserve attributes
    ds_matched.attrs = ds_ERA5.attrs
    ds_matched.attrs["longitude_range_used"] = f"±{lon_range} degrees" if lon_range else "Nearest longitude point"
    ds_matched.attrs["latitude_range_used"] = f"±{lat_range} degrees" if lat_range else "Nearest latitude point"

    for var in ds_ERA5.variables:
        ds_matched[var].attrs = ds_ERA5[var].attrs

    return ds_matched


In [121]:
ds_matched = match_era5_to_glider(ds_glider, ds_ERA5, lon_range=None, lat_range=None)

In [122]:
ds_matched

<xarray.Dataset> Size: 77kB
Dimensions:         (PROFILE_NUMBER: 770)
Coordinates:
    valid_time      (PROFILE_NUMBER) datetime64[ns] 6kB 2008-06-06T18:00:00 ....
    latitude        (PROFILE_NUMBER) float64 6kB 61.5 61.5 61.5 ... 63.75 63.75
    longitude       (PROFILE_NUMBER) float64 6kB -8.25 -8.25 -8.25 ... -6.0 -6.0
    number          int64 8B 0
    expver          (PROFILE_NUMBER) <U4 12kB '0001' '0001' ... '0001' '0001'
  * PROFILE_NUMBER  (PROFILE_NUMBER) float64 6kB 1.0 2.0 3.0 ... 771.0 772.0
Data variables: (12/13)
    tp              (PROFILE_NUMBER) float32 3kB 0.0 0.0 ... 0.001427 0.001369
    slhf            (PROFILE_NUMBER) float32 3kB -2.926e+04 ... -1.623e+05
    ssr             (PROFILE_NUMBER) float32 3kB 8.218e+05 ... 8.243e+04
    str             (PROFILE_NUMBER) float32 3kB -1.594e+05 ... -5.322e+04
    sshf            (PROFILE_NUMBER) float32 3kB -1.889e+03 ... 1.677e+04
    e               (PROFILE_NUMBER) float32 3kB -1.17e-05 ... -6.492e-05
    ...              ...
    v10             (PROFILE_NUMBER) float32 3kB -1.709 -1.709 ... 7.905 8.293
    zust            (PROFILE_NUMBER) float32 3kB 0.08552 0.08552 ... 0.5988
    swh             (PROFILE_NUMBER) float32 3kB nan nan nan nan ... nan nan nan
    pp1d            (PROFILE_NUMBER) float32 3kB nan nan nan nan ... nan nan nan
    ust             (PROFILE_NUMBER) float32 3kB nan nan nan nan ... nan nan nan
    vst             (PROFILE_NUMBER) float32 3kB nan nan nan nan ... nan nan nan
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-01-31T16:47 GRIB to CDM+CF via cfgrib-0.9.1...
    longitude_range_used:    Nearest longitude point
    latitude_range_used:     Nearest latitude point

In [12]:
### cut the ds_all data to the time and the coordinates of the glider mission
# Convert ds1 values to numpy arrays
profile_lats = ds_glider.LATITUDE.groupby(ds_glider.PROFILE_NUMBER).mean().values
profile_lons = ds_glider.LONGITUDE.groupby(ds_glider.PROFILE_NUMBER).mean().values
profile_times = ds_glider.TIME.groupby(ds_glider.PROFILE_NUMBER).mean().values
# Find the nearest indices
print('One')
time_idx = np.searchsorted(ds_all.valid_time, profile_times)
lat_idx = np.searchsorted(ds_all.latitude, profile_lats)
lon_idx = np.searchsorted(ds_all.longitude, profile_lons)
print('Two')
# Ensure indices are within bounds
time_idx = np.clip(time_idx, 0, len(ds_all.valid_time) - 1)
lat_idx = np.clip(lat_idx, 0, len(ds_all.latitude) - 1)
lon_idx = np.clip(lon_idx, 0, len(ds_all.longitude) - 1)
print('Three')
# Select the nearest values
# Select the nearest values
nearest_ds2 = ds_all.isel(valid_time=time_idx, latitude=lat_idx, longitude=lon_idx)

# Flatten the dataset to 1D
ds_flat = nearest_ds2.stack(all_points=("valid_time", "latitude", "longitude")).reset_index("all_points")

# Convert to dataset with only 1 dimension
ds_flat = ds_flat.to_dataframe().reset_index().set_index("all_points").to_xarray()

One
Two
Three


In [13]:
ds_flat

<xarray.Dataset> Size: 889MB
Dimensions:     (all_points: 8232000)
Coordinates:
  * all_points  (all_points) int64 66MB 0 1 2 3 ... 2743997 2743998 2743999
Data variables: (12/19)
    time        (all_points) int64 66MB 0 0 0 0 0 0 0 0 0 ... 2 2 2 2 2 2 2 2 2
    tp          (all_points) float32 33MB 0.0 0.0 0.0 0.0 ... nan nan nan nan
    slhf        (all_points) float32 33MB -2.525e+05 -2.525e+05 ... nan nan
    ssr         (all_points) float32 33MB 3.6e-12 3.6e-12 3.6e-12 ... nan nan
    str         (all_points) float32 33MB -1.414e+05 -1.414e+05 ... nan nan
    sshf        (all_points) float32 33MB -7.058e+04 -7.058e+04 ... nan nan
    ...          ...
    vst         (all_points) float32 33MB nan nan nan ... nan 0.00895 0.00895
    number      (all_points) int64 66MB 0 0 0 0 0 0 0 0 0 ... 0 0 0 0 0 0 0 0 0
    expver      (all_points) object 66MB '0001' '0001' '0001' ... '0001' '0001'
    valid_time  (all_points) datetime64[ns] 66MB 2008-02-14T23:00:00 ... 2008...
    latitude    (all_points) float64 66MB 61.75 61.75 61.75 ... 61.5 61.5 61.5
    longitude   (all_points) float64 66MB -3.0 -3.0 -3.0 ... -4.75 -4.5 -4.5